In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/somehappy/monthly/25-26 2nd term MT3 for students/Circulation Records for IB students of Pinghe Jinding in the academic year of 2024-2025 - all.xls
/kaggle/input/datasets/somehappy/monthly/25-26 2nd term MT3 for students/G10 Math 3rd Monthly Assessment Evaluation Criteria.pdf
/kaggle/input/datasets/somehappy/monthly/25-26 2nd term MT3 for students/~$Dataset.xlsx
/kaggle/input/datasets/somehappy/monthly/25-26 2nd term MT3 for students/2026 G10 3rd Monthly Assessment — Group Project.pdf
/kaggle/input/datasets/somehappy/monthly/25-26 2nd term MT3 for students/Circulation Records for IB students of Pinghe Jinding in the academic year of 2024-2025 - sample.xls


In [2]:
cp "/kaggle/input/datasets/somehappy/monthly/25-26 2nd term MT3 for students/Circulation Records for IB students of Pinghe Jinding in the academic year of 2024-2025 - all.xls" /kaggle/working/shit.xls

In [3]:
with open('/kaggle/input/datasets/somehappy/monthly/25-26 2nd term MT3 for students/Circulation Records for IB students of Pinghe Jinding in the academic year of 2024-2025 - all.xls', 'rb') as x:
    front = x.read(20)
    print(repr(front))

b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'


In [4]:
!pip install pandas
!pip install xlrd==1.2.0 openpyxl
!pip install pyxlsb
!pip install python-calamine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: xlrd
    Found existing installation: xlrd 2.0.2
    Uninstalling xlrd-2.0.2:
      Successfully uninstalled xlrd-2.0.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.1/935.1 kB 25.6 MB/s eta 0:00:00


In [5]:
with open("/kaggle/working/shit.xls", "rb") as f:
    header = f.read(16)
    print(header)

b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1\x00\x00\x00\x00\x00\x00\x00\x00'


In [6]:
from python_calamine import CalamineWorkbook

workbook = CalamineWorkbook.from_path("/kaggle/working/shit.xls")

# Get sheet names
# print(workbook.sheet_names) # Output: ["Sheet1", "Sheet2"]

# Get data from a specific sheet
sheet_data = workbook.get_sheet_by_name("Sheet1").to_python()

In [7]:
import pandas as pd
df = pd.DataFrame(sheet_data[1:], columns=sheet_data[0])
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3698 entries, 0 to 3697
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Collection Barcode     3698 non-null   object
 1   Title                  3698 non-null   object
 2   ISBN                   3698 non-null   object
 3   Circulation Type       3698 non-null   object
 4   Circulation Time       3698 non-null   object
 5   Price                  3698 non-null   object
 6   Author                 3698 non-null   object
 7   Publisher              3698 non-null   object
 8   Publication time       3698 non-null   object
 9   Place of Publication   3698 non-null   object
 10  Pages                  3698 non-null   object
 11  Classification Number  3698 non-null   object
 12  UserID                 3698 non-null   object
 13  Grade                  3698 non-null   object
dtypes: object(14)
memory usage: 404.6+ KB
None
  Collection Barcode         

In [8]:
df.to_csv("/kaggle/working/clean_data.csv", index=False, encoding='utf-8-sig')

Prompt: 好的，如何把一个.csv文件的第二行删了？

Response: 在 Pandas 中，CSV 文件的“第二行”通常指的是**索引为 1 的那一行数据**（因为第一行通常是表头/列名，被视为第 0 行数据之前的元数据；如果是指数据内容的第二行，那就是索引 1）。

以下是几种常用方法，推荐使用方法一。

### 方法一：使用 `drop` 删除指定索引行（最推荐）

```python
import pandas as pd

file_path = "your_data.csv"  # 替换为你的文件名

# 1. 读取 CSV
try:
    df = pd.read_csv(file_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding='gbk')

# 2. 删除索引为 1 的行（即数据的第二行）
# 注意：如果文件没有表头，header=None，则第二行数据索引也是 1
df_deleted = df.drop(index=1)

# 3. 重置索引（可选，让索引从 0 开始连续排列）
df_deleted = df_deleted.reset_index(drop=True)

# 4. 保存回原文件或新文件
df_deleted.to_csv(file_path, index=False, encoding='utf-8-sig')

print("已删除第二行并保存。")
```

### 方法二：读取时跳过第二行（适用于大文件，更高效）

如果你不需要加载整个文件到内存再删除，可以在读取时直接跳过特定行。

```python
import pandas as pd

file_path = "your_data.csv"

# skiprows 可以接受一个列表，表示要跳过的行索引（从0开始计数）
# 假设第一行是表头（index 0），第二行数据是 index 1
# 注意：skiprows 中的索引是相对于文件物理行的。
# 如果 header=0 (默认)，则物理第2行（索引1）是第一条数据。

df = pd.read_csv(file_path, skiprows=[1], encoding='utf-8-sig')

# 保存
df.to_csv(file_path, index=False, encoding='utf-8-sig')
print("已跳过第二行并保存。")
```

### ⚠️ 重要提示：关于“第二行”的定义

1. **如果“第二行”指的是数据部分的第二行**：
   - 使用上面的代码即可（`index=1` 或 `skiprows=[1]`）。

2. **如果“第二行”指的是文件物理上的第二行（可能是表头的一部分或空行）**：
   - 如果第一行是表头，第二行是数据 → 用上面代码。
   - 如果第一行和第二行都是表头（罕见），你需要先确认结构。
   - 如果第二行是空行或注释行，`skiprows=[1]` 同样有效。

3. **如果文件没有表头（header=None）**：
   - 第一行数据索引是 0，第二行数据索引是 1。
   - 读取时需加 `header=None`，删除时仍用 `drop(index=1)`。

### 如何验证是否删对了？

在保存前，可以先打印前后几行看看：

```python
print("删除前的前5行：")
print(df.head())

df_deleted = df.drop(index=1).reset_index(drop=True)

print("\n删除后的前5行：")
print(df_deleted.head())
```

确认无误后再执行 `to_csv` 保存。

Model: Qwen3.5:397b-a17b

In [9]:
import pandas as pd
file_path = "/kaggle/working/clean_data.csv"
df = pd.read_csv(file_path, skiprows=[1], encoding='utf-8-sig')
df.to_csv(file_path, index=False, encoding='utf-8-sig')
print("✅ 已删除物理第2行（中文表头）并保存。")

✅ 已删除物理第2行（中文表头）并保存。


model(gpt-4.1) out put: 
你完全可以用一个小模型（比如几行Python脚本配合pandas或正则表达式）来统一CSV里**Publication time**这一列的日期格式。下面是一个简单且高效的方案，能把混杂“年”、“年月日”和异常内容全部处理成统一的`YYYY-MM-DD`格式，如果只有年份，就补成`YYYY-01-01`。

假设你的CSV叫做`data.csv`，内容类似于：

```
Publication time
2015-07-15
2009-11-24
2020-09-03
...
 2021
Books,
```

直接贴代码，管用！

```python name=normalize_publication_time.py
import pandas as pd
import re

def normalize_date(val):
    # 去除多余空格
    val = str(val).strip()
    # 只保留数字和-，去掉Books,或者其它异常
    if not re.match(r'^\d{4}(-\d{2}(-\d{2})?)?$', val):
        # 如果包含年份，但带其他东西
        found = re.findall(r'(\d{4})', val)
        if found:
            val = found[0]
        else:
            return ''
    # 只年份
    if re.match(r'^\d{4}$', val):
        return f"{val}-01-01"
    # 年月
    if re.match(r'^\d{4}-\d{2}$', val):
        return f"{val}-01"
    # 年月日
    if re.match(r'^\d{4}-\d{2}-\d{2}$', val):
        return val
    # 其它情况
    return ''

df = pd.read_csv('data.csv')
df['Publication time'] = df['Publication time'].apply(normalize_date)
df.to_csv('data_normalized.csv', index=False)
```

### 说明
- 只保留格式为`YYYY-MM-DD`、`YYYY-MM`、`YYYY`的输入，其余（如“Books,”、“2021Books,”、“2023abc”）都过滤或按照年份兜底。
- **只有“年”时自动补全为“YYYY-01-01”。**
- 有恶心玩意儿比如“Books,”直接给处理成空字符串；要是混在“2022Books,”也会抽出“2022”。

### 完全不用“模型”（machine learning），正则和pandas够了。

如有增强需求、特殊格式，补充例子我给你加！

如果还要用**更智能的“小模型”**（比如transformers、LLM之类），其实完全没必要，除非你遇到极端难搞的历史古籍格式或者多语言混杂。

需要这段Python脚本的升级版？或者要直接在命令行操作也行，继续说！
User input: 怎么把一个csv的Publication time这个列里面日期的格式统一？2015-07-15
2009-11-24
2020-09-03
2010-10-05
1993-10-26
 2015
2016-09-12
 2021
2023
2023
2020
2016
2022
2021
Books,
2021
Books,
用一个他妈的小模型行么？

In [10]:
import pandas as pd
import re

def normalize_date(val):
    # 去除多余空格
    val = str(val).strip()
    # 只保留数字和-，去掉Books,或者其它异常
    if not re.match(r'^\d{4}(-\d{2}(-\d{2})?)?$', val):
        # 如果包含年份，但带其他东西
        found = re.findall(r'(\d{4})', val)
        if found:
            val = found[0]
        else:
            return ''
    # 只年份
    if re.match(r'^\d{4}$', val):
        return f"{val}-01-01"
    # 年月
    if re.match(r'^\d{4}-\d{2}$', val):
        return f"{val}-01"
    # 年月日
    if re.match(r'^\d{4}-\d{2}-\d{2}$', val):
        return val
    # 其它情况
    return ''

df = pd.read_csv('/kaggle/working/clean_data.csv')
df['Publication time'] = df['Publication time'].apply(normalize_date)
df.to_csv('data_normalized.csv', index=False)
print("傻逼")


傻逼


In [11]:
import pandas as pd
import re

def clean_pages(s):
    s = str(s)
    if not s.strip():
        return ''
    # 全角转半角
    s = s.translate(str.maketrans('０１２３４５６７８９，', '0123456789,'))
    # 去掉空格、回车
    s = s.replace(' ', '').replace('\n', '')
    # 千分位逗号直接去
    s = s.replace(',', '')
    # 提取所有连续数字串
    nums = re.findall(r'\d+', s)
    if nums:
        # 只认最大位数的数字（比如17,222页就是17222，不认17/222，只认大的）
        return str(max(map(int, nums)))
    return ''

df = pd.read_csv('/kaggle/working/data_normalized.csv')
df['Pages'] = df['Pages'].apply(clean_pages)
df.to_csv('/kaggle/working/katrina.csv', index=False)
print("大傻逼")

大傻逼


In [12]:
import pandas as pd

# ================= 配置区域 =================
# 请在这里填入你的 CSV 文件路径
CSV_FILE = "/kaggle/working/katrina.csv"  # <--- 修改这里为你的文件名
COLUMN_NAME = "Circulation Type"
# ===========================================

def check_invalid_data():
    valid_types = {"归还", "外借", "续借"}
    
    try:
        # 尝试读取文件
        try:
            df = pd.read_csv(CSV_FILE, encoding='utf-8-sig')
        except UnicodeDecodeError:
            df = pd.read_csv(CSV_FILE, encoding='gbk')
            
        if COLUMN_NAME not in df.columns:
            print(f"❌ 错误: 列 '{COLUMN_NAME}' 不存在于文件中。")
            print(f"可用列: {df.columns.tolist()}")
            return

        # 处理数据：转字符串、去空格
        # fillna('') 确保空值不会导致报错，且空值会被视为无效（因为 '' 不在合法集合中）
        series = df[COLUMN_NAME].fillna('').astype(str).str.strip()
        
        # 找出无效数据
        is_invalid = ~series.isin(valid_types)
        invalid_rows = df[is_invalid]
        
        if invalid_rows.empty:
            print("✅ 所有数据均符合规范。")
        else:
            print(f"⚠️ 发现 {len(invalid_rows)} 条无效数据：")
            print("-" * 30)
            # 只打印索引和该列内容，清晰明了
            print(invalid_rows[[COLUMN_NAME]])
            
    except FileNotFoundError:
        print(f"❌ 错误: 找不到文件 '{CSV_FILE}'，请检查路径是否正确。")
    except Exception as e:
        print(f"❌ 发生错误: {e}")

if __name__ == "__main__":
    check_invalid_data()

✅ 所有数据均符合规范。


In [13]:
pip install chardet

Note: you may need to restart the kernel to use updated packages.


In [14]:
import pandas as pd
def process_damn_data(Input, Output):
    df = pd.read_csv(Input);
    df.columns = df.columns.astype(str).str.strip()

    time_col=None;
    if 'Circulation Time' in df.columns:
        time_col='Circulation Time';
    elif 'Beginning of borrow' in df.columns:
        time_col='Beginning of borrow';
    else:
        print("Columns:", list(df.columns))
        raise KeyError("Need 'Circulation Time' in raw csv (or 'Beginning of borrow' if you passed a processed file).")

    df[time_col]=pd.to_datetime(df[time_col], errors='coerce')
    df = df.sort_values(by=['Collection Barcode', time_col]).reset_index(drop=True);

    processed_records=[]
    grouped=df.groupby('Collection Barcode')
    for barcode, group in grouped:
        current_record=None
        for _, row in group.iterrows():
            ctype=str(row.get('Circulation Type',''))

            if "外借" in ctype and "续借" not in ctype:
                if current_record is not None: 
                    current_record['Finally returned']="False"
                    current_record['Return time']=None;
                    current_record['Total time']=pd.NA;
                    processed_records.append(current_record)

                current_record=row.to_dict()
                current_record['Beginning of borrow']=row[time_col]
                current_record['Number of borrow']=0;
                current_record['Return time']=None;
                current_record['Finally returned']="False"
                current_record['Total time']=pd.NA;

            elif "续借" in ctype: 
                if current_record is not None:
                    current_record['Number of borrow']+=1

            elif "归还" in ctype:
                if current_record is not None:
                    current_record["Return time"]=row[time_col]
                    current_record['Finally returned']="True"
                    current_record['Total time']=current_record['Return time']-current_record['Beginning of borrow'];
                    processed_records.append(current_record)
                    current_record=None

        if current_record is not None: 
            current_record['Finally returned']="False"
            current_record['Return time']=None;
            current_record['Total time']=pd.NA;
            processed_records.append(current_record)

    result_df=pd.DataFrame(processed_records)
    new_cols=['Beginning of borrow', 'Return time', 'Total time', 'Number of borrow', 'Finally returned']
    original_cols = [c for c in df.columns if c not in new_cols and c != time_col]
    result_df = result_df[original_cols + new_cols]
    result_df = result_df.sort_values(by=['Beginning of borrow']).reset_index(drop=True);
    result_df.to_csv(Output, index=False, encoding='utf-8-sig')
    print("done")
if __name__=="__main__": 
    process_damn_data('/kaggle/working/katrina.csv','/kaggle/working/katrina_processed.csv')

done


In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from itertools import combinations

df = pd.read_csv("/kaggle/working/katrina.csv")
output_dir = "/kaggle/working/diagrams_without_outliers"
os.makedirs(output_dir, exist_ok=True)

df_encoded = df.copy()

for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df_encoded[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        unique_vals = sorted(df[col].dropna().unique())
        mapping = {v: i for i, v in enumerate(unique_vals)}
        df_encoded[col] = df[col].map(mapping)

def remove_outliers_iqr(s, k=1.5):
    s = pd.to_numeric(s, errors='coerce')
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        return pd.Series([True] * len(s), index=s.index)
    low = q1 - k * iqr
    high = q3 + k * iqr
    return s.between(low, high) | s.isna()

for col_x, col_y in combinations(df.columns, 2):
    x = df_encoded[col_x]
    y = df_encoded[col_y]

    mask_x = remove_outliers_iqr(x, k=1.5)
    mask_y = remove_outliers_iqr(y, k=1.5)
    mask = mask_x & mask_y

    plt.figure(figsize=(6, 4))
    plt.scatter(x[mask], y[mask], alpha=0.6)
    plt.xlabel(col_x)
    plt.ylabel(col_y)
    plt.title(f"{col_x} vs {col_y} (outliers removed)")
    filename = f"{col_x}__{col_y}.png".replace("/", "_")
    plt.savefig(os.path.join(output_dir, filename), dpi=150)
    plt.close()

print("done")

done


In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from itertools import combinations
df = pd.read_csv("/kaggle/working/katrina.csv")
output_dir = "/kaggle/working/diagrams"
os.makedirs(output_dir, exist_ok=True)
df_encoded = df.copy()
for col in df.columns:
    if df[col].dtype == "object":
        unique_vals = sorted(df[col].dropna().unique())
        mapping = {v: i for i, v in enumerate(unique_vals)}
        df_encoded[col] = df[col].map(mapping)
for col_x, col_y in combinations(df.columns, 2):
    plt.figure(figsize=(6, 4))
    plt.scatter(df_encoded[col_x], df_encoded[col_y], alpha=0.6)
    plt.xlabel(col_x)
    plt.ylabel(col_y)
    plt.title(f"{col_x} vs {col_y}")
    filename = f"{col_x}__{col_y}.png".replace("/", "_")
    plt.savefig(os.path.join(output_dir, filename), dpi=150)
    plt.close()
print("done at /kaggle/working/diagrams/")


done at /kaggle/working/diagrams/


In [17]:
!tar -cvf shit.tar /kaggle/working/diagrams_without_outliers

tar: Removing leading `/' from member names
/kaggle/working/diagrams_without_outliers/
/kaggle/working/diagrams_without_outliers/ISBN__Place of Publication.png
/kaggle/working/diagrams_without_outliers/Publisher__Classification Number.png
/kaggle/working/diagrams_without_outliers/Circulation Time__Publication time.png
/kaggle/working/diagrams_without_outliers/Publication time__Classification Number.png
/kaggle/working/diagrams_without_outliers/Title__Circulation Time.png
/kaggle/working/diagrams_without_outliers/ISBN__Publisher.png
/kaggle/working/diagrams_without_outliers/Circulation Time__Publisher.png
/kaggle/working/diagrams_without_outliers/ISBN__Author.png
/kaggle/working/diagrams_without_outliers/Title__Grade.png
/kaggle/working/diagrams_without_outliers/Collection Barcode__Circulation Type.png
/kaggle/working/diagrams_without_outliers/Circulation Type__Grade.png
/kaggle/working/diagrams_without_outliers/Author__Place of Publication.png
/kaggle/working/diagrams_without_outliers/

In [18]:
!tar -cvf damn.tar /kaggle/working/diagrams

tar: Removing leading `/' from member names
/kaggle/working/diagrams/
/kaggle/working/diagrams/ISBN__Place of Publication.png
/kaggle/working/diagrams/Publisher__Classification Number.png
/kaggle/working/diagrams/Circulation Time__Publication time.png
/kaggle/working/diagrams/Publication time__Classification Number.png
/kaggle/working/diagrams/Title__Circulation Time.png
/kaggle/working/diagrams/ISBN__Publisher.png
/kaggle/working/diagrams/Circulation Time__Publisher.png
/kaggle/working/diagrams/ISBN__Author.png
/kaggle/working/diagrams/Title__Grade.png
/kaggle/working/diagrams/Collection Barcode__Circulation Type.png
/kaggle/working/diagrams/Circulation Type__Grade.png
/kaggle/working/diagrams/Author__Place of Publication.png
/kaggle/working/diagrams/Circulation Time__Pages.png
/kaggle/working/diagrams/ISBN__Circulation Time.png
/kaggle/working/diagrams/Circulation Time__Grade.png
/kaggle/working/diagrams/Publication time__Grade.png
/kaggle/working/diagrams/Circulation Time__Classific